# Podejście C — Fine-tuning HerBERT

**Praca magisterska: Detekcja nadużyć w czacie platformy marketplace**

Model: `allegro/herbert-base-cased` (polski BERT, Allegro)

Klasy: `bypass` | `fraud` | `toxic` | `benign`

### Instrukcja:
1. Runtime → Change runtime type → **T4 GPU**
2. Uruchom komórki po kolei (Shift+Enter)
3. W komórce **[3]** wgraj 3 pliki: `train.jsonl`, `val.jsonl`, `test.jsonl`
4. Po zakończeniu pobierz `herbert.json` i wgraj do `praca-magisterska/results/metrics/`

In [ ]:
# [1] Instalacja zależności
!pip install -q --upgrade transformers datasets scikit-learn accelerate sacremoses

import transformers
print(f"transformers: {transformers.__version__}")

In [ ]:
# [2] Opcjonalnie: montowanie Google Drive (do zapisu modelu)
# Możesz pominąć — model pobierzesz ręcznie pod koniec
SAVE_TO_DRIVE = False   # zmień na True jeśli chcesz zapisać model na Dysk

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/stylify-herbert'
    print(f'Model będzie zapisany do: {DRIVE_DIR}')
else:
    DRIVE_DIR = None
    print('Tryb bez Drive — model zapisany lokalnie w Colab (/content/herbert-finetuned)')

In [ ]:
# [3] Wgranie plików datasetu
# Wgraj: train.jsonl, val.jsonl, test.jsonl
# Znajdziesz je w: praca-magisterska/data/processed/
from google.colab import files
uploaded = files.upload()
print('Wgrane pliki:', list(uploaded.keys()))

In [ ]:
# [4] Wczytanie datasetu
import json
from collections import Counter

LABELS   = ['bypass', 'fraud', 'toxic', 'benign']
LABEL2ID = {l: i for i, l in enumerate(LABELS)}
ID2LABEL = {i: l for i, l in enumerate(LABELS)}

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train_data = load_jsonl('train.jsonl')
val_data   = load_jsonl('val.jsonl')
test_data  = load_jsonl('test.jsonl')

for name, data in [('train', train_data), ('val', val_data), ('test', test_data)]:
    print(f'{name}: {len(data)} przykładów | {dict(Counter(d["label"] for d in data))}')

In [ ]:
# [5] Tokenizacja
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = 'allegro/herbert-base-cased'
MAX_LENGTH = 128   # wiadomości czatu są krótkie

print(f'Ładowanie tokenizera: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer gotowy.')

class AbuseDataset(Dataset):
    def __init__(self, data):
        self.encodings = tokenizer(
            [d['text'] for d in data],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH,
        )
        self.labels = [LABEL2ID[d['label']] for d in data]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = AbuseDataset(train_data)
val_dataset   = AbuseDataset(val_data)
test_dataset  = AbuseDataset(test_data)
print(f'Datasety gotowe: train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}')

In [ ]:
# [6] Ładowanie modelu HerBERT
from transformers import AutoModelForSequenceClassification

print(f'Ładowanie modelu: {MODEL_NAME} ...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Urządzenie: {device}')
if device == 'cpu':
    print('⚠️  UWAGA: Brak GPU — trening potrwa kilka godzin. Zmień runtime na T4 GPU!')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# [7] Fine-tuning
import numpy as np
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'macro_f1':    f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'accuracy':    float((preds == labels).mean()),
    }

training_args = TrainingArguments(
    output_dir='/content/herbert-finetuned',
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    report_to='none',
    fp16=(device == 'cuda'),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('Rozpoczynam fine-tuning HerBERT...')
trainer.train()
print('Trening zakończony!')

In [ ]:
# [8] Ewaluacja na zbiorze testowym
from sklearn.metrics import classification_report, confusion_matrix

preds_out  = trainer.predict(test_dataset)
y_pred_ids = np.argmax(preds_out.predictions, axis=1)
y_true_ids = [LABEL2ID[d['label']] for d in test_data]

y_pred = [ID2LABEL[p] for p in y_pred_ids]
y_true = [ID2LABEL[t] for t in y_true_ids]

report = classification_report(
    y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0
)
cm = confusion_matrix(y_true, y_pred, labels=LABELS).tolist()

fn_examples = [
    {'text': test_data[i]['text'], 'true': y_true[i], 'pred': y_pred[i]}
    for i in range(len(y_true))
    if y_pred[i] == 'benign' and y_true[i] != 'benign'
][:5]
fp_examples = [
    {'text': test_data[i]['text'], 'true': y_true[i], 'pred': y_pred[i]}
    for i in range(len(y_true))
    if y_pred[i] != 'benign' and y_true[i] == 'benign'
][:5]

result = {
    'detector': 'Podejscie C - HerBERT fine-tuned (allegro/herbert-base-cased)',
    'n_test': len(test_data),
    'per_class': {
        lbl: {
            'precision': round(report[lbl]['precision'], 4),
            'recall':    round(report[lbl]['recall'],    4),
            'f1':        round(report[lbl]['f1-score'],  4),
            'support':   report[lbl]['support'],
        }
        for lbl in LABELS
    },
    'macro_f1':         round(report['macro avg']['f1-score'],    4),
    'weighted_f1':      round(report['weighted avg']['f1-score'], 4),
    'accuracy':         round(report['accuracy'],                 4),
    'confusion_matrix': cm,
    'confusion_labels': LABELS,
    'n_errors_fp':      len(fp_examples),
    'n_errors_fn':      len(fn_examples),
    'examples_fp':      fp_examples,
    'examples_fn':      fn_examples,
    # Predykcje per-przykład (potrzebne do ewaluacji podejścia D — Hybrid B+C)
    'y_pred': y_pred,
    'y_true': y_true,
}

print('=' * 60)
print(f"  {result['detector']}")
print('=' * 60)
print(f"  Accuracy:    {result['accuracy']:.3f}")
print(f"  Macro F1:    {result['macro_f1']:.3f}")
print(f"  Weighted F1: {result['weighted_f1']:.3f}")
print()
print(f"  {'Klasa':<10} {'Precision':>10} {'Recall':>10} {'F1':>10} {'N':>6}")
print('  ' + '-' * 48)
for lbl in LABELS:
    pc = result['per_class'][lbl]
    print(f"  {lbl:<10} {pc['precision']:>10.3f} {pc['recall']:>10.3f} {pc['f1']:>10.3f} {int(pc['support']):>6}")
print()
print('Macierz pomyłek (wiersze=prawdziwe, kolumny=przewidziane):')
print(f"  {'':12}" + ''.join(f'{l:>10}' for l in LABELS))
for i, lbl in enumerate(LABELS):
    print(f"  {lbl:<12}" + ''.join(f'{v:>10}' for v in cm[i]))

In [ ]:
# [9] Zapis wyników + pobranie
import json, shutil
from google.colab import files

# Zapisz metryki (ten plik wgraj do results/metrics/herbert.json)
with open('herbert.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print('herbert.json zapisany.')

# Zapisz model na Dysk Google (jeśli włączone)
if SAVE_TO_DRIVE and DRIVE_DIR:
    import os
    os.makedirs(DRIVE_DIR, exist_ok=True)
    trainer.save_model(DRIVE_DIR)
    tokenizer.save_pretrained(DRIVE_DIR)
    print(f'Model zapisany na Google Drive: {DRIVE_DIR}')
    print('Pobierz folder i wgraj do: praca-magisterska/models/herbert-finetuned/')

# Pobierz herbert.json bezpośrednio
files.download('herbert.json')
print('\nGotowe! Wgraj herbert.json do:')
print('  praca-magisterska/results/metrics/herbert.json')
print('Następnie uruchom lokalnie:')
print('  python experiments/evaluation/evaluate_approaches.py')

In [ ]:
# [10] Pobierz wytrenowany model do wdrożenia w ML service
# Uruchom po zakończeniu treningu — pobierze zip z plikami modelu.
# Następnie: rozpakuj i wgraj zawartość do ml-service/models/herbert-finetuned/
import shutil
from google.colab import files

print('Zapisywanie najlepszego modelu...')
trainer.save_model('/content/herbert-finetuned')
tokenizer.save_pretrained('/content/herbert-finetuned')

print('Pakowanie do ZIP...')
shutil.make_archive('herbert-finetuned', 'zip', '/content/herbert-finetuned')

print('Pobieranie herbert-finetuned.zip...')
files.download('herbert-finetuned.zip')
print()
print('Co dalej:')
print('  1. Rozpakuj herbert-finetuned.zip')
print('  2. Skopiuj zawartość do: ml-service/models/herbert-finetuned/')
print('  3. docker-compose restart ml-service')